# Evaluating Recommendations on Real Cohort Data
### Sprint 2, Item 4

Runs the full matching pipeline across the entire cleaned cohort (all resumes, all
job descriptions) rather than a single example category, and computes real accuracy
metrics using each resume's actual `Category` label as ground truth: for a given
resume, is its own category the top-ranked (or top-3-ranked) job description by
similarity?

Compares TF-IDF, embeddings, and (if run) zero-shot classification side by side.

Assumes `tfidf_baseline.ipynb` and `embeddings.ipynb` (Sprint 1) have both been run.


In [3]:
%run config.ipynb

Config loaded. Data will be stored under: c:\Users\RanVic\Downloads\applicant_screening_sprint2\applicant_screening_nb\data_pipeline\data


## 1. Load data and saved artifacts

In [4]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics.pairwise import cosine_similarity

resumes_clean = pd.read_csv(PROCESSED_RESUMES_CSV)
job_descriptions = pd.read_csv(JOB_DESCRIPTIONS_CSV)

TFIDF_DIR = PROCESSED_DATA_DIR / "tfidf"
EMBEDDINGS_DIR = PROCESSED_DATA_DIR / "embeddings"

# --- Required before loading the saved vectorizer ---
# tfidf_baseline.ipynb saved a TfidfVectorizer that uses a custom tokenizer
# function (tokenizer_joiner, built on tokenize). Pickle only stores a
# *reference* to that function's name, not its code — so it must be
# redefined here, identically, before joblib can unpickle the vectorizer.
import re
import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

STOPWORDS = set(stopwords.words("english"))

def tokenize(text: str) -> list[str]:
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = text.split()
    return [t for t in tokens if t not in STOPWORDS and len(t) > 2]

def tokenizer_joiner(text: str) -> list[str]:
    return tokenize(text)
# --- end required redefinition ---

tfidf_vectorizer = joblib.load(TFIDF_DIR / "tfidf_vectorizer.joblib")
tfidf_resume_vectors = joblib.load(TFIDF_DIR / "resume_vectors.joblib")
tfidf_job_vectors = joblib.load(TFIDF_DIR / "job_vectors.joblib")

resume_embeddings = np.load(EMBEDDINGS_DIR / "resume_embeddings.npy")
job_embeddings = np.load(EMBEDDINGS_DIR / "job_embeddings.npy")

print(f"Evaluating on {len(resumes_clean)} resumes across {len(job_descriptions)} job categories")

Evaluating on 2481 resumes across 24 job categories


## 2. Compute full similarity matrices
One row per resume, one column per job description — for both methods.

In [5]:
tfidf_sim_matrix = cosine_similarity(tfidf_resume_vectors, tfidf_job_vectors)
embedding_sim_matrix = resume_embeddings @ job_embeddings.T  # pre-normalized -> dot == cosine

print("TF-IDF similarity matrix:", tfidf_sim_matrix.shape)
print("Embedding similarity matrix:", embedding_sim_matrix.shape)

TF-IDF similarity matrix: (2481, 24)
Embedding similarity matrix: (2481, 24)


## 3. Top-1 and Top-3 accuracy
For each resume, check whether its true category appears as the #1 or within the
top-3 highest-similarity job descriptions.

In [6]:
job_categories = job_descriptions[COL_CATEGORY].tolist()

def top_k_accuracy(sim_matrix: np.ndarray, k: int) -> float:
    correct = 0
    for i, true_category in enumerate(resumes_clean[COL_CATEGORY]):
        top_k_idx = np.argsort(sim_matrix[i])[::-1][:k]
        top_k_categories = [job_categories[idx] for idx in top_k_idx]
        if true_category in top_k_categories:
            correct += 1
    return correct / len(resumes_clean)

metrics = pd.DataFrame({
    "method": ["TF-IDF", "Embeddings"],
    "top_1_accuracy": [
        top_k_accuracy(tfidf_sim_matrix, 1),
        top_k_accuracy(embedding_sim_matrix, 1),
    ],
    "top_3_accuracy": [
        top_k_accuracy(tfidf_sim_matrix, 3),
        top_k_accuracy(embedding_sim_matrix, 3),
    ],
})
metrics

,method,top_1_accuracy,top_3_accuracy
0,TF-IDF,0.634019,0.849254
1,Embeddings,0.661427,0.817009


## 4. Per-category breakdown
Overall accuracy can hide categories the model handles poorly — this breaks it down
per job category so weak spots are visible.

In [7]:
def per_category_accuracy(sim_matrix: np.ndarray) -> pd.DataFrame:
    rows = []
    for category in job_categories:
        mask = resumes_clean[COL_CATEGORY] == category
        if mask.sum() == 0:
            continue
        sub_matrix = sim_matrix[mask.values]
        sub_resumes = resumes_clean[mask]
        correct = 0
        for i, true_category in enumerate(sub_resumes[COL_CATEGORY]):
            top_idx = np.argmax(sub_matrix[i])
            if job_categories[top_idx] == true_category:
                correct += 1
        rows.append({
            "category": category,
            "n_resumes": mask.sum(),
            "top_1_accuracy": correct / mask.sum(),
        })
    return pd.DataFrame(rows).sort_values("top_1_accuracy")

print("Weakest categories (embeddings method):")
per_category_accuracy(embedding_sim_matrix).head(10)

Weakest categories (embeddings method):


,category,n_resumes,top_1_accuracy
4,ARTS,103,0.174757
5,AUTOMOBILE,36,0.222222
3,APPAREL,97,0.329897
2,AGRICULTURE,63,0.412698
7,BANKING,115,0.443478
14,DIGITAL-MEDIA,96,0.531250
21,PUBLIC-RELATIONS,111,0.531532
13,DESIGNER,107,0.532710
17,FITNESS,117,0.547009
6,AVIATION,116,0.560345


---
## Notes
- Ground truth here is the dataset's own `Category` label being matched against a job
  description *derived from that same category's vocabulary* — so high accuracy
  partly reflects the fact that the job descriptions were built directly from resume
  text in `preprocess.ipynb`. This is a reasonable sanity check, but not the same as
  evaluating against real hiring outcomes or independently written job postings.
- The per-category breakdown is the more actionable output: categories with
  low top-1 accuracy likely share vocabulary with a neighboring category (e.g.
  different engineering disciplines) and are worth inspecting individually.
